# Gemma 4 E2B Text SFT — Resmi Unsloth Pattern (2026)

Bu notebook resmi `Gemma4_(E2B)_Text.ipynb` notebook'unun birebir uyarlamasıdır. `01_sft_modern.ipynb` (Qwen3) ile yan yana okumak için tasarlandı.

## Qwen3 vs Gemma 4 — Pratik Farklar

| | Qwen3-4B-Instruct | Gemma 4 E2B-it |
|---|---|---|
| **Class** | `FastLanguageModel` | **`FastModel`** (multimodal-native) |
| **PEFT API** | `target_modules=[...]` listesi | **`finetune_*` flag'leri** (vision/lang/attn/mlp) |
| **LoRA** | r=32, alpha=32 | **r=8, alpha=8** (model küçük) |
| **Chat template** | `qwen3-instruct` | **`gemma-4`** |
| **Markerlar** | `<\|im_start\|>user\n` / `<\|im_start\|>assistant\n` | **`<\|turn>user\n` / `<\|turn>model\n`** |
| **Format quirk** | yok | **`removeprefix('<bos>')`** zorunlu |
| **Generation** | T=0.7, top_p=0.8, top_k=20 | **T=1.0, top_p=0.95, top_k=64** |

**Donanım:** 16GB GPU yeterli (Gemma 4 E2B = ~2B activated params)

## 1. Setup

In [ ]:
import unsloth                                # MUTLAKA EN BAŞTA
from unsloth import FastModel                  # FastLanguageModel DEĞİL — Gemma 4 multimodal-native
from unsloth.chat_templates import (
    get_chat_template,
    standardize_data_formats,
    train_on_responses_only,
)
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 2. Model — Gemma 4 E2B Instruct

Resmi listeden seçenekler:

**Instruct (chat-tuned):**
- `unsloth/gemma-4-E2B-it` — ~2B activated, 16GB GPU'da rahat
- `unsloth/gemma-4-E4B-it` — ~4B activated
- `unsloth/gemma-4-31B-it` — büyük
- `unsloth/gemma-4-26B-A4B-it` — MoE

**Base (pre-train, instruction-following yok):**
- `unsloth/gemma-4-E2B`
- `unsloth/gemma-4-E4B`

**Önemli:** `dtype = None` — Unsloth otomatik bf16/fp16 seçer. `load_in_4bit = False` — 16-bit LoRA Gemma 4 E2B için 16GB'a sığar.

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,                           # auto-detect (bf16 if Ampere+)
    max_seq_length = 1024,                   # uzun context için artır
    load_in_4bit = False,                   # 16-bit LoRA (16GB'a sığar). True = 4-bit QLoRA
    full_finetuning = False,                # [NEW!] full FT artık var
    # token = "YOUR_HF_TOKEN",              # gated modeller için
)

## 3. LoRA — `finetune_*` Flag'leri

Gemma 4 multimodal-native olduğu için `FastModel.get_peft_model` Qwen3'ten farklı:

- `target_modules` listesi **yok**
- Yerine 4 adet `finetune_*` boolean flag var
- Text-only SFT için `finetune_vision_layers = False`
- r ve alpha küçük (8) — model E2B (~2B activated)

**RSLoRA / DoRA / LoftQ** isterseniz `use_rslora = True` veya `loftq_config = ...` ekleyin (resmi notebook bunları kullanmıyor).

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,    # Text-only SFT — vision tower'i dondur
    finetune_language_layers   = True,     # LM ana gövde — açık
    finetune_attention_modules = True,     # Attention — GRPO için de iyi
    finetune_mlp_modules       = True,     # MLP — her zaman açık

    r = 8,                                  # Larger = higher accuracy ama overfit riski
    lora_alpha = 8,                         # alpha == r önerilir
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## 4. Chat Template — `gemma-4`

Gemma 4 markerları **Qwen3'ten farklı**:

```
<bos><|turn>user
Soru<turn|>
<|turn>model
Cevap<turn|>
```

Asimetrik markerlara dikkat: açılış `<|turn>` ama kapanış `<turn|>`.

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

# Doğrula
sample_msgs = [
    {"role": "user", "content": "Faiz nedir?"},
    {"role": "assistant", "content": "Faiz, paranin zaman degeridir."},
]
formatted = tokenizer.apply_chat_template(sample_msgs, tokenize=False)
print(formatted)

## 5. Veri — FineTome-100k (resmi notebook'taki dataset)

Maxime Labonne'un FineTome-100k ShareGPT format datasetini kullanıyoruz. İlk 3000 satır.

**Akış:** raw → `standardize_data_formats` (ShareGPT/Alpaca → conversations) → `formatting_prompts_func` (chat template uygula + `<bos>` strip)

In [ ]:
dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:3000]")
print(f'Raw rows: {len(dataset)}')

# standardize_data_formats: ShareGPT/Alpaca → 'conversations' kolonu
dataset = standardize_data_formats(dataset)
print(dataset[0])

### `formatting_prompts_func` — `<bos>` strip ZORUNLU

Chat template `<bos>` ekler ama processor de eğitim öncesi `<bos>` ekleyecek. Çift `<bos>` model'i bozar — `removeprefix('<bos>')` ile temizliyoruz.

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
        for c in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"][:400])

## 6. SFTTrainer

Resmi notebook ile birebir aynı ayarlar:
- `weight_decay = 0.001` (resmi default — 0.01 değil)
- `lr_scheduler_type = "linear"` (cosine değil)
- `warmup_steps = 5`
- `optim = "adamw_8bit"`
- `seed = 3407`

**Smoke test:** `max_steps = 60` (production: `num_train_epochs = 1`)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,       # effective batch = 4
        warmup_steps = 5,
        max_steps = 60,                         # demo; production: num_train_epochs=1
        learning_rate = 2e-4,                   # LoRA için. Long training: 2e-5
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,                   # 0.001 — resmi default
        lr_scheduler_type = "linear",           # linear — resmi default
        seed = 3407,
        report_to = "none",
    ),
)

## 7. `train_on_responses_only` — Gemma 4 Markerları

**ÖNEMLİ:** Gemma 4 markerları Qwen3'ten farklı:
- Qwen3: `<|im_start|>user\n` / `<|im_start|>assistant\n`
- **Gemma 4: `<|turn>user\n` / `<|turn>model\n`** (asistan rolü `model`)

Yanlış marker kullanırsan masking sessizce hiçbir şey yapmaz — full sequence üzerinde loss hesaplar (kullanıcı input'unu da öğrenir, kötü sonuç). Decode ile mutlaka kontrol et.

In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",     # Gemma 4 markerı
    response_part    = "<|turn>model\n",    # Gemma 4 markerı (assistant değil — 'model'!)
)

# Masking dogrulama — 100. ornek
sample_idx = min(100, len(trainer.train_dataset) - 1)
print("=== FULL INPUT (instruction + response) ===")
print(tokenizer.decode(trainer.train_dataset[sample_idx]["input_ids"]))

print("\n=== ONLY UNMASKED LABELS (sadece response gorunmeli) ===")
print(tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in trainer.train_dataset[sample_idx]["labels"]
]).replace(tokenizer.pad_token, " "))

## 8. Training

In [ ]:
# Memory snapshot — resmi notebook'lardaki gibi
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"\n{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

## 9. Inference

Gemma 4 önerilen generation parametreleri: `temperature=1.0, top_p=0.95, top_k=64`.

Gemma 4 multimodal olduğu için input format Qwen3'ten farklı — `content` listesi `{"type": "text", "text": ...}` formatında olmalı (text-only kullanım için bile).

In [ ]:
from transformers import TextStreamer

messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "Continue the sequence: 1, 1, 2, 3, 5, 8,"}],
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,           # ZORUNLU — generation icin
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature = 1.0, top_p = 0.95, top_k = 64,    # Gemma 4 önerisi
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

## 10. Save & Deploy

In [ ]:
# A. LoRA adapter (en kucuk)
model.save_pretrained("gemma4_e2b_lora")
tokenizer.save_pretrained("gemma4_e2b_lora")

# B. Merged 16-bit (vLLM/HF inference)
# model.save_pretrained_merged(
#     "gemma4_e2b_merged",
#     tokenizer,
#     save_method = "merged_16bit",
# )

# C. GGUF (Ollama/llama.cpp)
# model.save_pretrained_gguf(
#     "gemma4_e2b_gguf",
#     tokenizer,
#     quantization_method = "q4_k_m",
# )

# D. Hub'a push
# model.push_to_hub("USER/gemma4_e2b_lora", token="hf_xxx")

print("LoRA saved: gemma4_e2b_lora/")

## Önemli Notlar

1. **`<bos>` strip** — `formatting_prompts_func` içinde `removeprefix('<bos>')` zorunlu. Yoksa çift bos = bozuk model.
2. **Marker'lar Qwen3'ten farklı** — `<|turn>user\n` / `<|turn>model\n`. Asistan rolü 'assistant' değil **'model'**.
3. **`tools=` desteği YOK** — Gemma 4 chat template'inde `tools` branch'i yok. Tool calling istiyorsan system prompt'a manuel JSON schema göm.
4. **`enable_thinking`** — `gemma-4-thinking` template'i ayrı (bu notebook standart Gemma 4 kullanıyor).
5. **`finetune_vision_layers = False`** — text-only için zorunlu, yoksa vision encoder da güncellenir (gereksiz VRAM + zarar).
6. **Generation `content` formatı** — `[{"type": "text", "text": ...}]` listesi. Düz string vermek hata verebilir.